In [1]:
import pandas as pd
import re

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

In [2]:
hpt_df = pd.read_csv('../takehome/data/hpt_extract_20250213.csv')
tic_df = pd.read_csv('../takehome/data/tic_extract_20250213.csv')

# EDA
## Hospital data

In [3]:
hpt_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2950 entries, 0 to 2949
Data columns (total 22 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   source_file_name                       2950 non-null   object 
 1   hospital_id                            2950 non-null   object 
 2   hospital_name                          2950 non-null   object 
 3   last_updated_on                        2950 non-null   object 
 4   hospital_state                         2950 non-null   object 
 5   license_number                         2950 non-null   object 
 6   payer_name                             2950 non-null   object 
 7   plan_name                              2950 non-null   object 
 8   code_type                              2950 non-null   object 
 9   raw_code                               2950 non-null   object 
 10  description                            2950 non-null   object 
 11  sett

In [4]:
hpt_df.isnull().sum()/hpt_df.shape[0]

source_file_name                         0.000000
hospital_id                              0.000000
hospital_name                            0.000000
last_updated_on                          0.000000
hospital_state                           0.000000
license_number                           0.000000
payer_name                               0.000000
plan_name                                0.000000
code_type                                0.000000
raw_code                                 0.000000
description                              0.000000
setting                                  0.000000
modifiers                                0.988475
standard_charge_gross                    0.319661
standard_charge_discounted_cash          0.315254
standard_charge_negotiated_dollar        0.144407
standard_charge_negotiated_percentage    0.772881
standard_charge_min                      0.787119
standard_charge_max                      0.787119
standard_charge_methodology              0.072542


In [5]:
hpt_binary_features = [column for column in hpt_df.columns if hpt_df[column].dropna().isin([0, 1]).all() 
                   and hpt_df[column].nunique() == 2]

hpt_categorical_features = [column for column in hpt_df.columns if hpt_df[column].dtype in ["object", "category"] 
                        and column not in hpt_binary_features]

hpt_numeric_features = [column for column in hpt_df.columns if column not in hpt_binary_features and column not in hpt_categorical_features]

In [6]:
print(f"Binary: {hpt_binary_features}")
print(f"Categorical: {hpt_categorical_features}")
print(f"Numerical: {hpt_numeric_features}")

Binary: []
Categorical: ['source_file_name', 'hospital_id', 'hospital_name', 'last_updated_on', 'hospital_state', 'license_number', 'payer_name', 'plan_name', 'code_type', 'raw_code', 'description', 'setting', 'modifiers', 'standard_charge_methodology', 'additional_payer_notes', 'additional_generic_notes']
Numerical: ['standard_charge_gross', 'standard_charge_discounted_cash', 'standard_charge_negotiated_dollar', 'standard_charge_negotiated_percentage', 'standard_charge_min', 'standard_charge_max']


In [7]:
# What's the cardinality of these categorical columns?
hpt_df[hpt_categorical_features].nunique()

source_file_name                 3
hospital_id                      3
hospital_name                    3
last_updated_on                  3
hospital_state                   1
license_number                   3
payer_name                      73
plan_name                      538
code_type                        3
raw_code                         4
description                     20
setting                          3
modifiers                        1
standard_charge_methodology     15
additional_payer_notes          85
additional_generic_notes         5
dtype: int64

In [8]:
{col: hpt_df[col].unique() for col in hpt_categorical_features if "list" not in col}

{'source_file_name': array(['13-1740114_montefiore-medical-center_standardcharges.csv.gz',
        '131624096_mount-sinai-hospital_standardcharges.csv.gz',
        '133971298-1801992631_nyu-langone-tisch_standardcharges.csv.gz'],
       dtype=object),
 'hospital_id': array(['62915ae8-8d64-4e2f-b05f-b18edde57a3d',
        '5954cbad-a7c5-43f7-b356-8f2ecdad579a',
        '40e6a8c8-a68c-4d28-b1d5-fa70d6d09636'], dtype=object),
 'hospital_name': array(['Montefiore Medical Center', 'The Mount Sinai Hospital',
        'NYU Langone'], dtype=object),
 'last_updated_on': array(['2024-07-01', '2024-09-16', '2025-01-01'], dtype=object),
 'hospital_state': array(['NY'], dtype=object),
 'license_number': array(['13-1740114', '330024', '7002053H'], dtype=object),
 'payer_name': array(['Aetna', 'HealthFirst', 'Cigna', 'Oscar', 'Healthcare', 'United',
        'Emblem', 'Humana', 'Fidelis', 'Centerlight', 'Partners',
        'Northwell', 'Agewell', 'MetroPlus', 'Longevity', 'MVP',
        'Hamaspik', 'W

### Observations from hospital data
- Payer names seem messy, upper/lowercase mixed, different versions like UHC, United healthcare etc
- 3 hospitals only, that's nice and focused
- Plan name is messy, abbreviations, concatenated versions etc
- Raw code is given as string, because there is MS-DRG 872 there. Should prob take code type out of that one and make it all numeric, which is how it seems to be in payer data
- For most rows, modifiers missing. When populated it's only HK
- Generic notes seems to be more like
- Code type LOCAL seems to have CPT-type code and only 43239 and only from NYU. I don't know what "LOCAL" is but since it looks like CPT, probably normalize?

### Pre-processing

In [9]:
# Let's just take out the code type out of here
hpt_df.loc[hpt_df.raw_code == "MS-DRG 872", "raw_code"] = "872"

# now code can be int, like in payer
hpt_df["raw_code"] = hpt_df["raw_code"].astype(int)

# lowercase for normalization
hpt_df["payer_name"] = hpt_df["payer_name"].str.lower()

# There are only three payers in the payer dataset and there is limited ambiguity around those names in the hospital dataset
# so we could use a simple canonical mapping for normalization
# When we go to scale with this, we would need to get smarter about this, maybe a combination canonical mapping,
# Levenshtein distance, checking if one name is completely included by another name etc
payer_map = {
    "unitedhealthcare": ["uhc", "united", "united healthcare"],
    "cigna-corporation": ["cigna"],
    "aetna": ["aetna"]
    }

for canonical_payer_name, payer_names in payer_map.items():
    hpt_df.loc[hpt_df["payer_name"].isin(payer_names), "payer_name"] = canonical_payer_name
# now payer_names in hospital dataset are canonicalized and match the payer dataset
# It would probably also be a good idea to put this canonical version into a separate column in the larger analysis,
# in order to preserve the original, for troubleshooting problems with canonicalization

# LOCAL comes only from NYU and looks like CPT, so I'll just normalize it.
# This may not hold when we scale this, maybe this is a different way hospitals do it?
def normalize_code_type(code_type, raw_code):
    if code_type == 'LOCAL' and re.match(r'^\d{5}$', str(raw_code)):
        return 'CPT'
    return code_type

hpt_df.rename(columns = {"code_type": "code_type_original"}, inplace = True)

hpt_df['code_type'] = hpt_df.apply(
    lambda r: normalize_code_type(r['code_type_original'], r['raw_code']),
    axis=1
)

## Payer data

In [10]:
tic_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 222 entries, 0 to 221
Data columns (total 17 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   payer                       222 non-null    object 
 1   network_name                222 non-null    object 
 2   network_id                  222 non-null    object 
 3   network_year_month          222 non-null    int64  
 4   network_region              222 non-null    object 
 5   code                        222 non-null    int64  
 6   code_type                   222 non-null    object 
 7   ein                         222 non-null    int64  
 8   taxonomy_filtered_npi_list  218 non-null    object 
 9   modifier_list               7 non-null      object 
 10  billing_class               222 non-null    object 
 11  place_of_service_list       197 non-null    object 
 12  negotiation_type            222 non-null    object 
 13  arrangement                 222 non

In [11]:
tic_df.isnull().sum()/tic_df.shape[0]

payer                         0.000000
network_name                  0.000000
network_id                    0.000000
network_year_month            0.000000
network_region                0.000000
code                          0.000000
code_type                     0.000000
ein                           0.000000
taxonomy_filtered_npi_list    0.018018
modifier_list                 0.968468
billing_class                 0.000000
place_of_service_list         0.112613
negotiation_type              0.000000
arrangement                   0.000000
rate                          0.000000
cms_baseline_schedule         0.004505
cms_baseline_rate             0.004505
dtype: float64

In [12]:
tic_binary_features = [column for column in tic_df.columns if tic_df[column].dropna().isin([0, 1]).all() 
                   and tic_df[column].nunique() == 2]

tic_categorical_features = [column for column in tic_df.columns if tic_df[column].dtype in ["object", "category"] 
                        and column not in tic_binary_features]

tic_numeric_features = [column for column in tic_df.columns if column not in tic_binary_features and column not in tic_categorical_features]

In [13]:
print(f"Binary: {tic_binary_features}")
print(f"Categorical: {tic_categorical_features}")
print(f"Numerical: {tic_numeric_features}")

Binary: []
Categorical: ['payer', 'network_name', 'network_id', 'network_region', 'code_type', 'taxonomy_filtered_npi_list', 'modifier_list', 'billing_class', 'place_of_service_list', 'negotiation_type', 'arrangement', 'cms_baseline_schedule']
Numerical: ['network_year_month', 'code', 'ein', 'rate', 'cms_baseline_rate']


In [14]:
# Cardinality of categorical features
tic_df[tic_categorical_features].nunique()

payer                           3
network_name                    3
network_id                      3
network_region                  1
code_type                       2
taxonomy_filtered_npi_list    106
modifier_list                   2
billing_class                   2
place_of_service_list          12
negotiation_type                3
arrangement                     1
cms_baseline_schedule          18
dtype: int64

In [15]:
{col: tic_df[col].unique() for col in tic_categorical_features + ["code"] if "list" not in col}

{'payer': array(['unitedhealthcare', 'aetna', 'cigna-corporation'], dtype=object),
 'network_name': array(['choice-plus', 'open-access-managed-choice', 'national-oap'],
       dtype=object),
 'network_id': array(['592bc118-0dac-4f38-949c-11dc9b3a3879',
        '39f0d406-b5df-4046-9759-f08565e45db7',
        '5dbd8f1c-3f56-4806-917b-e495668bf2bf'], dtype=object),
 'network_region': array(['USA'], dtype=object),
 'code_type': array(['MS-DRG', 'CPT'], dtype=object),
 'billing_class': array(['institutional', 'professional'], dtype=object),
 'negotiation_type': array(['negotiated', 'fee schedule', 'percentage'], dtype=object),
 'arrangement': array(['ffs'], dtype=object),
 'cms_baseline_schedule': array(['IPPS', 'PFS_NONFACILITY_1320201', 'PFS_NONFACILITY_1320202',
        'PFS_NONFACILITY_1320203', 'PFS_NONFACILITY_0610216', 'OPPS',
        'PFS_NONFACILITY_1240201', 'PFS_FACILITY_1320202',
        'PFS_NONFACILITY_1329204', 'PFS_FACILITY_NPA',
        'PFS_NONFACILITY_0111263', 'PFS_FACIL

### Observations from Payer data
- Three payers only, nice and focused
- There is no link to the hospital, but maybe network name is linkable to plan name in hospital data?
- Code is numeric
- No LOCAL as code type
- There are two "list-like" columns, one is place of service, the other one with NPIs

# How do we link payer dataset to hospitals?

- It looks like EIN (Employer Identification Number) is a link here.
    - Payer dataset has "ein" and the "source_file_name" in hospital dataset starts with EIN. This was not clear but there were 3 distinct source file names, 3 distinct EINs and I looked up those numbers in source file names and confirmed that those are indeed associated with those hospitals
    - For the sake of simplicity, I will have a simple map of hospital name, as it appears in hospital dataset to a list of EINs. NYU Langone seems to have two EINs, hence a list.
    - For scaling purposes, we would either need to build a lookup table (is probably a maintenance headache) or use a public API like from Propublica (https://projects.propublica.org/nonprofits/api) to query EIN. Or both! Build a lookup table using the API, update only for new hospitals that are not found in the lookup table.
- Hospital data has "standard_charge_methodology" that would be relevant to "negotiation_type" in the payer data. The latter has 'negotiated', 'fee schedule', 'percentage', the former has related things like "case rate" and "fee schedule". Some normalization will be needed between the two.
- So what are the explicit keys to join on?
    - Hospital name
    - code type
    - code
    - normalized charge methodology
- I would then give a "confidence score" to each match based on whether the settings make sense, charge methodology makes sense, whether we have info from both hospital and payer or just from one source etc

## Map hospital name through EIN

In [16]:
ein_map = {
    "Montefiore Medical Center": [131740114],
    "The Mount Sinai Hospital": [131624096],
    "NYU Langone": [133971298, 1801992631]
}

for hospital_name, eins in ein_map.items():
    tic_df.loc[tic_df["ein"].isin(eins), "hospital_name"] = hospital_name

## Normalize billing/charging methodology

In [17]:
tic_methodology_map = {
    'negotiated': 'negotiated_dollar',
    'fee schedule': 'fee_schedule',
    'percentage': 'percentage',
}

hpt_methodology_map = {
    'case rate': 'negotiated_dollar',
    'fee schedule': 'fee_schedule',
    'percent of total billed charges': 'percentage',
}

In [18]:
hpt_df.standard_charge_methodology.value_counts()
# hospital data has some that are not in the map above. keep those as is.

standard_charge_methodology
Case Rate                          1316
percent of total billed charges     613
fee schedule                        322
Fee Schedule                        224
case rate                            81
Case rate                            80
Percent of total billed charges      51
other                                34
Other                                 8
5228                                  2
per diem                              1
93157.15                              1
11671                                 1
32829.5                               1
17412.69                              1
Name: count, dtype: int64

In [19]:
# What's up with the numeric stuff in standard_charge_methodology?
# Looks like all of those are from NYU Langone
# Probably some data extraction mishap
# I will guess that a field shifted during export. Not sure where exactly this shift happened.
# There are only a handful of these and I think we leave these as is, instead of trying to fix it.
# This data issue should be flagged downstream
hpt_df.loc[hpt_df.standard_charge_methodology.isin(["5228", "93157.15", "11671", "32829.5", "17412.69"])]

,source_file_name,hospital_id,hospital_name,last_updated_on,hospital_state,license_number,payer_name,plan_name,code_type_original,raw_code,description,setting,modifiers,standard_charge_gross,standard_charge_discounted_cash,standard_charge_negotiated_dollar,standard_charge_negotiated_percentage,standard_charge_min,standard_charge_max,standard_charge_methodology,additional_payer_notes,additional_generic_notes,code_type
463,133971298-1801992631_nyu-langone-tisch_standardcharges.csv.gz,40e6a8c8-a68c-4d28-b1d5-fa70d6d09636,NYU Langone,2025-01-01,NY,7002053H,hip,emblemessential3and4(cohort2hcp)3901,CPT,99283,HC EMERG DEPT F/U VISIT LVL 3,both,NaN,2182.13,414.60,NaN,1.00,NaN,NaN,5228,NaN,NaN,CPT
534,133971298-1801992631_nyu-langone-tisch_standardcharges.csv.gz,40e6a8c8-a68c-4d28-b1d5-fa70d6d09636,NYU Langone,2025-01-01,NY,7002053H,hip,emblemessential3and4(cohort2hcp)3901,MS-DRG,872,SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOURS WITHOUT MCC,inpatient,NaN,NaN,NaN,NaN,10076.87,NaN,NaN,93157.15,NaN,NaN,MS-DRG
835,133971298-1801992631_nyu-langone-tisch_standardcharges.csv.gz,40e6a8c8-a68c-4d28-b1d5-fa70d6d09636,NYU Langone,2025-01-01,NY,7002053H,hip,emblemessential3and4(cohort2hcp)3901,CPT,43239,HC UGI W BX. SGL/MULTIPLE,both,NaN,1464.39,278.23,NaN,446.75,NaN,NaN,11671,NaN,NaN,CPT
985,133971298-1801992631_nyu-langone-tisch_standardcharges.csv.gz,40e6a8c8-a68c-4d28-b1d5-fa70d6d09636,NYU Langone,2025-01-01,NY,7002053H,hip,emblemessential3and4(cohort2hcp)3901,CPT,99283,HC EMERGENCY DEPT VISIT LVL 3,both,NaN,5106.17,970.17,NaN,1.00,NaN,NaN,5228,NaN,NaN,CPT
1059,133971298-1801992631_nyu-langone-tisch_standardcharges.csv.gz,40e6a8c8-a68c-4d28-b1d5-fa70d6d09636,NYU Langone,2025-01-01,NY,7002053H,hip,emblemessential3and4(cohort2hcp)3901,LOCAL,43239,HEAD HUM 19MM 50MM SHLDR UNIVERS STRL LF CUF ARTHROPATHY,both,NaN,32829.50,6237.61,NaN,2462.21,NaN,NaN,32829.5,NaN,NaN,CPT
2604,133971298-1801992631_nyu-langone-tisch_standardcharges.csv.gz,40e6a8c8-a68c-4d28-b1d5-fa70d6d09636,NYU Langone,2025-01-01,NY,7002053H,hip,emblemessential3and4(cohort2hcp)3901,CPT,43239,EGD BIOPSY SINGLE/MULTIPLE,both,NaN,NaN,NaN,NaN,446.75,NaN,NaN,17412.69,NaN,NaN,CPT


In [20]:
hpt_df["standard_charge_methodology"] = hpt_df["standard_charge_methodology"].str.lower()

hpt_df["normalized_methodology"] = hpt_df["standard_charge_methodology"].map(
    {methodology: normalized for methodology, normalized in hpt_methodology_map.items()}
).fillna(hpt_df["standard_charge_methodology"])

tic_df["normalized_methodology"] = tic_df["negotiation_type"].map(
    {methodology: normalized for methodology, normalized in tic_methodology_map.items()}
).fillna(tic_df["negotiation_type"])

## Plan name in hospital data vs network name in payer data

In [21]:
# Apparently these are all PPO
tic_df.network_name.unique()

array(['choice-plus', 'open-access-managed-choice', 'national-oap'],
      dtype=object)

In [22]:
# There are plans that are PPO, HMO, EPO and then there are plans that are not clear
sorted(hpt_df.loc[hpt_df.payer_name.isin(tic_df.payer.unique())]["plan_name"].unique())

['ASA',
 'All Payer',
 'All Payers',
 'Blackstone, Student Health, Savings Plus',
 'Commercial',
 'Empire Plan',
 'HC Compass',
 'HC NY Essential',
 'HC Oxford',
 'HMO and PPO',
 'HMO/POS/PPO',
 'Lifesource',
 'LocalPlus',
 'Medicare',
 'Medicare Advantage',
 'Medicare Advantage, Community Plan Medicare Advantage',
 'Signature Administrators',
 'aetnaglobalnationaltransplant(hbonly)1437',
 'aetnahmo1005',
 'aetnaindemnity1006',
 'aetnameritainhealth1444',
 'aetnaopenaccesselectchoice(epo)1415',
 'aetnaopenaccesshmo1414',
 'aetnapos1009',
 'aetnappo1010',
 'aetnasignatureadministratorsppo1417',
 'aetnastudentplan1008',
 'allied3025',
 'chesterfieldresourcesinc1641',
 'christianbrothers1640',
 'cignaevernorthbehavioralhealth1490',
 'cignagreatwestppo1442',
 'cignahmo1403',
 'cignaindemnity1045',
 'cignalifesource(hbonly)1678',
 'cignamanagedcare/pos1046',
 'cignanalc1448',
 'cignaopenaccess1413',
 'cignaopenaccesshmo(hbonly)1466',
 'cignaopenaccessppo1465',
 'cignappo1047',
 'cignapremie

### Plan confidence scoring

In [23]:
# I think I would want to start simple with this and assign high confidence to matches where plan name clearly indicates a PPO,
# a slightly lower confidence to matches that might be commercial/PPO and low confidence to rest.
# Something like
def plan_confidence(plan_name):
    if pd.isna(plan_name):
        return 0.5  # uncertain
    p = plan_name.lower().replace('-', '').replace(' ', '')
    if any(x in p for x in ['medicare', 'medicaid', 'advantage']):
        return 0.1  # incompatible
    if any(x in p for x in ['ppo', 'choice', 'openaccess', 'oap']):
        return 1.0  # ppo_match
    if any(x in p for x in ['allpayer', 'allpayers', 'commercial']):
        return 0.9  # commercial_catchall
    return 0.5      # uncertain

## What's with setting in the hospital dataset and place of service in the payer dataset?

In [24]:
hpt_df.setting.value_counts()
# this seems to indicate what kind of setting the rate applies to
# Earlier I saw that this is always populated

setting
both          2208
inpatient      408
outpatient     334
Name: count, dtype: int64

In [25]:
tic_df["place_of_service_list"].value_counts()

# These place of service codes correspond descriptions here: https://www.cms.gov/medicare/coding-billing/place-of-service-codes/code-sets

place_of_service_list
01,02,03,04,05,06,07,08,10,11,12,13,14,15,17,18,19,20,21,22,23,24,25,26,27,31,32,33,34,41,42,49,50,51,52,53,54,55,56,57,58,60,61,62,65,71,72,81,99       56
01,02,03,04,05,06,07,08,09,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,31,32,33,34,41,42,49,50,51,52,53,54,55,56,57,58,60,61,62,65,71,72,81,99    30
11                                                                                                                                                       26
01,03,04,05,06,07,08,10,11,12,13,14,15,17,18,20,25,27,32,33,49,50,54,55,57,58,60,62,65,71,72,81,99                                                       23
02,19,21,22,23,24,26,31,34,41,42,51,52,53,56,61                                                                                                          23
01,06,08,12,19,20,22,23,24,25,26,27,41,42,50,52,53,57,58,60,62,65,99                                                                                     22
21,31,32,33,34,51,54,55,56,61             

In [26]:
# I think I can use these to assess alignment with "setting" in the hospital dataset, i.e. inpatient, outpatient, both
# I could have an "inferred" setting in payer data looking for codes that are clearly inpatient and codes that are clearly outpatient
# I thought at first I would do something like: if the list contains more than a number of codes, then it's sort of "catch all" or has both inpatient or outpatient settings
# But it looks like there are some with many codes that are only inpatient ("21,31,32,33,34,51,54,55,56,61")
# I decided to point Claude Sonnet 4.6 to the CMS website and ask to generate sets of relatively unambiguously inpatient and outpatient codes
# I will use these to determine if a list has, in addition to potentially ambiguous codes, only inpatient or outpatient codes or both.

inpatient_codes = {
    '21',  # Inpatient Hospital — patients admitted for medical conditions
    '31',  # Skilled Nursing Facility — primarily inpatient skilled nursing care
    '32',  # Nursing Facility — primarily skilled nursing care to residents
    '33',  # Custodial Care Facility — long-term residential, room and board
    '34',  # Hospice — palliative care facility (not patient's home)
    '51',  # Inpatient Psychiatric Facility — inpatient psychiatric, 24-hour basis
    '54',  # Intermediate Care Facility/Individuals with Intellectual Disabilities — residential
    '55',  # Residential Substance Abuse Treatment Facility — live-in residents
    '56',  # Psychiatric Residential Treatment Center — 24-hour group living
    '61',  # Comprehensive Inpatient Rehabilitation Facility — inpatient physical rehab
}

outpatient_codes = {
    '11',  # Office — ambulatory basis, not a hospital or facility
    '17',  # Walk-in Retail Health Clinic — ambulatory, preventive/primary care
    '19',  # Off Campus-Outpatient Hospital — do not require hospitalization
    '20',  # Urgent Care Facility — ambulatory, unscheduled immediate care
    '22',  # On Campus-Outpatient Hospital — do not require hospitalization
    '24',  # Ambulatory Surgical Center — surgical services on ambulatory basis
    '49',  # Independent Clinic — outpatients only (explicitly stated in definition)
    '50',  # Federally Qualified Health Center — preventive primary care
    '57',  # Non-residential Substance Abuse Treatment Facility — ambulatory basis
    '58',  # Non-residential Opioid Treatment Facility — ambulatory basis
    '62',  # Comprehensive Outpatient Rehabilitation Facility — outpatients explicitly
    '65',  # End-Stage Renal Disease Treatment Facility — ambulatory or home-care basis
    '71',  # Public Health Clinic — ambulatory primary care
    '72',  # Rural Health Clinic — ambulatory primary care
}

In [27]:
def infer_payer_setting(place_of_service_list):
    if pd.isna(place_of_service_list):
        return 'unknown'
    
    codes = set(str(place_of_service_list).split(','))
    has_inpatient = bool(codes & inpatient_codes)
    has_outpatient = bool(codes & outpatient_codes)
    
    if has_inpatient and has_outpatient:
        return 'both'
    if has_inpatient:
        return 'inpatient'
    if has_outpatient:
        return 'outpatient'
    return 'ambiguous'

tic_df['inferred_setting'] = tic_df['place_of_service_list'].apply(infer_payer_setting)

### Setting confidence scoring

In [28]:
# I think I will use the inferred setting in the payer data to assess alignment with the hospital data
def setting_confidence(inferred_setting, hpt_setting):
    if pd.isna(inferred_setting) or pd.isna(hpt_setting):
        return 0.5 # uncertain
    
    setting_confidence = {
        ('inpatient',  'inpatient'): 1.0,
        ('outpatient', 'outpatient'): 1.0,
        ('both',       'both'): 1.0,
        ('both',       'inpatient'): 0.8,
        ('both',       'outpatient'): 0.8,
        ('inpatient',  'both'): 0.8,
        ('outpatient', 'both'): 0.8,
        ('ambiguous',  'inpatient'): 0.5,
        ('ambiguous',  'outpatient'): 0.5,
        ('ambiguous',  'both'): 0.5,
        ('unknown',    'inpatient'): 0.5,
        ('unknown',    'outpatient'): 0.5,
        ('unknown',    'both'): 0.5,
        ('inpatient',  'outpatient'): 0.2,
        ('outpatient', 'inpatient'): 0.2
    }
    
    return setting_confidence.get((inferred_setting, hpt_setting), 0.5)

# Align hospital data with payer data

In [29]:
# for alignment with the payer dataset
hpt_df.rename(columns = {"payer_name": "payer", "raw_code": "code"}, inplace=True)

join_columns = ["hospital_name", "payer", "code_type", "code", "normalized_methodology"]

# Doing full outer join to keep all records
rate_data_merged = pd.merge(hpt_df, tic_df, on = join_columns, how = "outer")

# Where does the match data come from?
rate_data_merged['match_source'] = 'both'
rate_data_merged.loc[rate_data_merged['rate'].isnull(), 'match_source'] = 'hospital_only'
rate_data_merged.loc[rate_data_merged['last_updated_on'].isnull(), 'match_source'] = 'payer_only'

# pick the correct hospital rate column depending on normalized methodology
def get_hpt_rate(row):
    if row['normalized_methodology'] == 'percentage':
        return row['standard_charge_negotiated_percentage']
    else:
        return row['standard_charge_negotiated_dollar']

rate_data_merged['hpt_rate'] = rate_data_merged.apply(get_hpt_rate, axis=1)

In [31]:
rate_data_merged['plan_score'] = None
rate_data_merged['setting_score'] = None
rate_data_merged['confidence'] = None
rate_data_merged['rate_pct_delta'] = None

both_mask = rate_data_merged['match_source'] == 'both'
has_rates = rate_data_merged['rate'].notna() & rate_data_merged['hpt_rate'].notna()
score_mask = both_mask & has_rates

# Plan and setting scores and confidence score are calculated only for source = both
rate_data_merged.loc[both_mask, 'plan_score'] = rate_data_merged.loc[both_mask].apply(
    lambda r: plan_confidence(r['plan_name']), axis=1
)
rate_data_merged.loc[both_mask, 'setting_score'] = rate_data_merged.loc[both_mask].apply(
    lambda r: setting_confidence(r['inferred_setting'], r['setting']), axis=1
)

rate_data_merged.loc[both_mask, 'confidence'] = (
    0.7 * rate_data_merged.loc[both_mask, 'setting_score'] +
    0.3 * rate_data_merged.loc[both_mask, 'plan_score']
)

In [32]:
# rate diff calculated only where we have a rate from both
rate_data_merged.loc[score_mask, 'rate_difference_pct'] = (
    100*(rate_data_merged.loc[score_mask, 'rate'] - rate_data_merged.loc[score_mask, 'hpt_rate']) /
    rate_data_merged.loc[score_mask, 'hpt_rate']
)

In [34]:
# what kind of rate match do we have
def match_tier(source, confidence, rate_pct_delta):
    if source == 'hospital_only':
        return 'no_matching_payer'
    if source == 'payer_only':
        return 'no_matching_hospital'
    # match_source = "both" below here
    if pd.isna(rate_pct_delta):
        return 'matched_but_no_rate'
    if confidence >= 0.8 and abs(rate_pct_delta) <= 5:
        return 'high_confidence_rate_agrees'
    if confidence >= 0.8 and abs(rate_pct_delta) > 5:
        return 'high_confidence_rate_disagrees'
    return 'low_confidence'

rate_data_merged['match_tier'] = rate_data_merged.apply(
    lambda r: match_tier(r['match_source'], r['confidence'], r['rate_difference_pct']), axis=1
)

In [35]:
data_schema_columns = [
    "match_source",
    "hospital_name",
    "payer_name",
    "payer_network_name",
    "hospital_plan_name",
    "payer_inferred_setting",
    "hospital_setting",
    "charge_methodology",
    "hospital_rate",
    "payer_rate",
    "rate_difference_pct",
    "confidence",
    "match_tier"
]

In [36]:
rate_data_merged.rename(columns = {
    "payer": "payer_name", # Should have kept it this way above
    "network_name": "payer_network_name",
    "plan_name": "hospital_plan_name",
    "inferred_setting": "payer_inferred_setting",
    "setting": "hospital_setting",
    "normalized_methodology": "charge_methodology",
    "hpt_rate": "hospital_rate",
    "rate": "payer_rate"
}, inplace = True)

In [37]:
unified_payer_data = rate_data_merged[data_schema_columns]